# 02 Status semantics
How each raw status behaves in consecutive months, and why C is mapped to PAID-OFF and X to DORMANT.

In [1]:
import logging, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("streamlit").setLevel(logging.ERROR)
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

In [2]:
from src.services import quality, rolls
m = quality.raw_transition_matrix()
m

2026-10-04 20:32:33.322 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:33.324 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:33.327 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


TO_STATUS,C,X,0,1,2,3,4,5
FROM_STATUS,,,,,,,,
C,419323,0,0,1,0,0,0,0
X,2035,162833,31553,452,36,6,8,41
0,19064,27156,319132,7057,103,16,1,44
1,998,397,5363,3412,559,6,0,2
2,64,36,210,137,138,257,7,0
3,20,16,26,14,22,28,185,3
4,27,8,11,0,3,2,22,145
5,68,47,23,4,7,5,0,1457


Row percentages: C is (almost) never left; X is left often, mainly back to 0.

In [3]:
(m.div(m.sum(axis=1), axis=0) * 100).round(2)

TO_STATUS,C,X,0,1,2,3,4,5
FROM_STATUS,,,,,,,,
C,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
X,1.03,82.67,16.02,0.23,0.02,0.00,0.00,0.02
0,5.12,7.29,85.66,1.89,0.03,0.00,0.00,0.01
1,9.29,3.70,49.95,31.78,5.21,0.06,0.00,0.02
2,7.54,4.24,24.73,16.14,16.25,30.27,0.82,0.00
3,6.37,5.10,8.28,4.46,7.01,8.92,58.92,0.96
4,12.39,3.67,5.05,0.00,1.38,0.92,10.09,66.51
5,4.22,2.92,1.43,0.25,0.43,0.31,0.00,90.44


In [4]:
met = quality.metrics()
print('C -> C:', int(met['c_to_c']), 'of', int(met['c_transitions_total']), '; C -> open:', int(met['c_to_open']))
print('dormant exiters who return:', round(quality.report()['dormant_return_share'] * 100, 1), '%')

C -> C: 419323 of 419324 ; C -> open: 1
dormant exiters who return: 53.9 %


## The source-literal reading is degenerate
Under C = CURRENT the CURRENT row essentially never deteriorates.

In [5]:
lit, summary = rolls.literal_sensitivity()
print(summary)
lit.pivot_table(index='FROM_STATE', columns='TO_STATE', values='p').round(4)

2026-10-04 20:32:33.377 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


{'current_persistence': 0.9999976152092416, 'current_deterioration': 2.3847907584588527e-06, 'current_row_n': 419324.0, 'literal_current_share_m0': 0.6539918306721129, 'evidence_b0_share_of_open_m0': 0.9566430564498819}


TO_STATE,1-29,120-149,30-59,60-89,90-119,CHARGE-OFF,CURRENT
FROM_STATE,,,,,,,
1-29,0.9239,0.0000,0.0204,0.0003,0.0000,0.0001,0.0552
120-149,0.0524,0.1048,0.0000,0.0143,0.0095,0.6905,0.1286
30-59,0.5187,0.0000,0.3300,0.0541,0.0006,0.0002,0.0965
60-89,0.2583,0.0086,0.1685,0.1697,0.3161,0.0000,0.0787
90-119,0.0872,0.6208,0.0470,0.0738,0.0940,0.0101,0.0671
CHARGE-OFF,0.0147,0.0000,0.0026,0.0045,0.0032,0.9316,0.0435
CURRENT,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000


## Charge-off departures (flagged, not fixed)

In [6]:
quality.anomalies()

,FROM_STATE,TO_STATE,n
0,CHARGE-OFF,PAID-OFF,68
1,CHARGE-OFF,DORMANT,47
2,CHARGE-OFF,1-29,23
3,CHARGE-OFF,60-89,7
4,CHARGE-OFF,90-119,5
5,CHARGE-OFF,30-59,4
6,PAID-OFF,30-59,1
